<a href="https://colab.research.google.com/github/Xcelrator0/Intership-Tasks/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
if not os.path.isdir("Intership-Tasks"):
    !git clone https://github.com/Xcelrator0/Intership-Tasks.git
%cd Intership-Tasks
%pip install -q pandas numpy scikit-learn

Cloning into 'Intership-Tasks'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 146 (delta 53), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 1.87 MiB | 10.16 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/Intership-Tasks


## 1. Ranked actions + reason codes


In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

BEST_MODEL_NAME = "random_forest"  # set to whatever your w05 comparison table actually crowned the winner

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

LABEL_SOURCE_COLS = {"trend_direction", "trend_pct", "is_declining_label"}
ID_COLS = {"content_id", "client_id"}
SUSPECT_LEAKAGE_COLS = {
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
}
EXCLUDE = LABEL_SOURCE_COLS | ID_COLS | SUSPECT_LEAKAGE_COLS
feature_cols = [c for c in df.columns if c not in EXCLUDE]
numeric_cols = [c for c in feature_cols if df[c].dtype != "object"]
categorical_cols = [c for c in feature_cols if df[c].dtype == "object"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx], df.iloc[test_idx]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
])
pipe = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(n_estimators=300, random_state=42))])
pipe.fit(train[feature_cols], train["is_declining_label"])

test = test.copy()
test["model_score"] = pipe.predict_proba(test[feature_cols])[:, 1]

def reason_code(row):
    if row["days_since_last_update"] >= df["days_since_last_update"].quantile(0.75):
        return "STALE_REFRESH_CANDIDATE"
    if row["ctr"] < df["ctr"].median() and row["avg_position"] <= 20:
        return "CTR_BELOW_EXPECTED"
    if row["model_score"] >= 0.7:
        return "MODEL_HIGH_RISK"
    return "MONITOR_ONLY"

def action_from_reason(code):
    return {
        "STALE_REFRESH_CANDIDATE": "refresh_content",
        "CTR_BELOW_EXPECTED": "fix_snippet_ctr",
        "MODEL_HIGH_RISK": "review_first",
        "MONITOR_ONLY": "monitor",
    }[code]

test["reason_code"] = test.apply(reason_code, axis=1)
test["action"] = test["reason_code"].apply(action_from_reason)

playbook_queue = test.sort_values("model_score", ascending=False)
out_cols = ["content_id", "model_score", "reason_code", "action", "days_since_last_update", "ctr", "avg_position"]
playbook_queue[out_cols].head(20)

,content_id,model_score,reason_code,action,days_since_last_update,ctr,avg_position
22042,content_2ba626fea4d6,0.973333,STALE_REFRESH_CANDIDATE,refresh_content,104,0.00,7.2
14343,content_9ac61c04930e,0.966667,STALE_REFRESH_CANDIDATE,refresh_content,104,0.22,5.6
21729,content_7766ffacdcfa,0.956667,STALE_REFRESH_CANDIDATE,refresh_content,104,0.00,9.3
22526,content_1d0963b56227,0.956667,STALE_REFRESH_CANDIDATE,refresh_content,104,0.09,39.0
6228,content_e988c1699454,0.956667,STALE_REFRESH_CANDIDATE,refresh_content,104,0.00,21.5
22730,content_bb1edbf0ba6c,0.950000,STALE_REFRESH_CANDIDATE,refresh_content,104,0.00,12.4
15732,content_6607847e4454,0.946667,STALE_REFRESH_CANDIDATE,refresh_content,104,0.13,9.3
17696,content_569475b335ab,0.946667,STALE_REFRESH_CANDIDATE,refresh_content,104,0.13,42.5
13215,content_98aa0aecb1d9,0.943333,STALE_REFRESH_CANDIDATE,refresh_content,104,0.06,5.1
10080,content_35d63627bf3e,0.940000,MODEL_HIGH_RISK,review_first,103,0.00,32.6


STALE_REFRESH_CANDIDATE -> refresh_content

Pages left untouched for over 100 days lose recency trust in search rankings. Updating the content restores freshness signals and stops organic traffic decay.

CTR_BELOW_EXPECTED -> fix_snippet_ctr

The page ranks in top search positions but fails to capture clicks. Rewriting the title tag and meta description improves snippet appeal without needing a full content rewrite.

MODEL_HIGH_RISK -> review_first

The model detects complex feature interactions signaling an impending decline, even though individual metrics look fine. An editor needs to manually inspect the page context before committing refresh resources.

MONITOR_ONLY -> monitor

Performance metrics and risk scores remain healthy. No manual intervention or budget allocation is needed right now.

## 2. Intended use and limits
This playbook is for content strategists deciding what to prioritize in a refresh sprint — it ranks pages by risk of decline and assigns a reason (stale, CTR-underperforming, or model-flagged), not a final verdict. It's decision-support, not an auto-publish system. It only covers pages in the same client/content mix as the Week-5 training data — client segments, content types, or languages not represented there shouldn't be scored with this model without re-validation. MODEL_HIGH_RISK picks (459 rows) come from the classifier alone, with no rule-based signal behind them, so they carry the least interpretability of the four reason codes.

## 3. Human review + the no-go list
Before acting on any refresh_content or review_first row, a person should confirm the page still matters to current business priorities — a stale page the model flags might already be scheduled for retirement or unrelated to current strategy. Never auto-publish content changes straight from action — every refresh still needs an editor. Should never be automated: anything touching legal, medical, or compliance-sensitive pages, and any page where ctr is 0.00 with very few real sessions (some top rows here show 0.00 CTR — worth checking these aren't just low-traffic noise before treating them as priority CTR fixes).

## 4. Monitoring / retrain triggers
Re-check the reason-code distribution monthly — if MONITOR_ONLY share drops sharply or MODEL_HIGH_RISK share spikes, the underlying traffic patterns have likely shifted since training. Re-measure precision@50 against real outcomes (did refresh_content pages actually recover?) at 60-90 days. Retrain if a new client segment or content type appears that wasn't in the Week-5 training data, since the model wasn't validated on it.

## 5. Exports for the paper

In [3]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

playbook_queue[out_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("wrote work/outputs/action_playbook_queue.csv")

import json
metrics = {
    "n_rows": int(len(playbook_queue)),
    "reason_code_counts": playbook_queue["reason_code"].value_counts().to_dict(),
    "action_counts": playbook_queue["action"].value_counts().to_dict(),
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/playbook_metrics.json — commit this, it's your paper's receipt")
metrics

wrote work/outputs/action_playbook_queue.csv
wrote work/outputs/playbook_metrics.json — commit this, it's your paper's receipt


{'n_rows': 6163,
 'reason_code_counts': {'MONITOR_ONLY': 2722,
  'CTR_BELOW_EXPECTED': 2016,
  'STALE_REFRESH_CANDIDATE': 966,
  'MODEL_HIGH_RISK': 459},
 'action_counts': {'monitor': 2722,
  'fix_snippet_ctr': 2016,
  'refresh_content': 966,
  'review_first': 459}}